In [1]:
import demes
import msprime
import numpy as np
import os

import h2py
from h2py.utils import timestamp

In [2]:
# Simulation parameters
L = 10_000_000
n_reps = 1
n_samples = 1  # per population
u = 1.5e-8
r = 1e-8
r_bins = np.logspace(-6, -2, 17)
pops = {"pop0": n_samples}
samples = ["tsk_0"]
depth = 5

# File names
pop_file = "data/populations.txt"
rec_map_file = "data/rec_map.txt"
bed_file = "data/coverage.bed"


def demographic_model():
    b = demes.Builder()
    b.add_deme("pop0", epochs=[dict(start_size=1e4, end_time=0)])
    g = b.resolve()
    return g


g = demographic_model()

In [3]:
def write_pop_file():
    with open(pop_file, "w") as fout:
        fout.write("sample\tpop\n")
        idx = 0
        for pop in samples:
            for sidx in range(n_samples):
                fout.write(f"tsk_{idx}\t{pop}\n")
                idx += 1


def write_rec_map_file():
    with open(rec_map_file, "w") as fout:
        fout.write("chrom\tPosition(bp)\tMap(cM)\n")
        fout.write("none\t0\t0\n")
        fout.write(f"none\t{L}\t{100*r*L}\n")


def write_bed_file():
    with open(bed_file, "w") as fout:
        fout.write(f"none\t0\t{L}\n")


write_pop_file()
write_rec_map_file()
write_bed_file()

In [4]:
def run_sim(g, i):
    demog = msprime.Demography.from_demes(g)
    ts = msprime.sim_ancestry(
        pops,
        demography=demog,
        sequence_length=L,
        recombination_rate=r,
    )
    ts = msprime.sim_mutations(ts, rate=u, model="binary")
    ref_seq = h2py.simulation.generate_sam_files(
            ts,
            samples,
            f"data/{{sample}}.{i}.sam",
            depth=depth,
            ref_name="reference",
            report=100000,
        )
    fasta_file = f"data/reference.{i}.fasta"
    h2py.utils.write_fasta_file(fasta_file, [ref_seq], ["reference"])
    print(timestamp(), f"Simulated replicate {i}")

for i in range(n_reps):
    run_sim(g, i)

[26-07-14 15:00:47] Wrote read 0; depth 3.7e-06
[26-07-14 15:00:49] Wrote read 100000; depth 0.4992965
[26-07-14 15:00:51] Wrote read 200000; depth 0.9988769
[26-07-14 15:00:54] Wrote read 300000; depth 1.4979123
[26-07-14 15:00:56] Wrote read 400000; depth 1.9974708
[26-07-14 15:00:59] Wrote read 500000; depth 2.4966779
[26-07-14 15:01:01] Wrote read 600000; depth 2.9966514
[26-07-14 15:01:04] Wrote read 700000; depth 3.496314
[26-07-14 15:01:06] Wrote read 800000; depth 3.9958959
[26-07-14 15:01:09] Wrote read 900000; depth 4.4955591
[26-07-14 15:01:11] Wrote read 999999; depth 4.9949678
[26-07-14 15:01:11] Finished writing data/tsk_0.0.sam
[26-07-14 15:01:13] Simulated replicate 0


In [6]:
# Use ATLAS to compute genotype probabilities
for i in range(n_reps):
    for sample in samples:
        !samtools view -b "data/{sample}.{i}.sam" | samtools sort -o "data/{sample}.{i}.bam"
        !atlas call --bam "data/{sample}.{i}.bam" --fasta "data/reference.{i}.fasta" --method Bayesian \
            --sampleName "{sample}" --out "data/{sample}.{i}" --noTriallelic

        !gzip -d "data/{sample}.{i}_calls_maximumAPosteriori.vcf.gz"
        !bgzip "data/{sample}.{i}_calls_maximumAPosteriori.vcf"
        !bcftools index "data/{sample}.{i}_calls_maximumAPosteriori.vcf.gz"

    args = " ".join([f"data/{sample}.{i}_calls_maximumAPosteriori.vcf.gz" for sample in samples])
    !bcftools merge -o "data/mergedGPs.{i}.vcf.gz" {args}

atlas: /lib/x86_64-linux-gnu/libhts.so.3: no version information available (required by atlas)

                   ATLAS 2.0.3
                  -------------

     https://bitbucket.org/wegmannlab/atlas

 Commit 7ab31971feabf9a9fd07409cedcc56d1b86e2d4d

   - Used executable: atlas
   - Interpreting 'call' as name of task.
   - Will write log to file 'data/tsk_0.0.log'. (use 'noLogFile' to supress or 'logFile' to specify the file-name)
   - Calling genotypes (task = call):
      - Writing output files with prefix 'data/tsk_0.0'. (parameter 'out')
      - Initializing random generator with seed 1'784'059'277'462. (use 'fixedSeed' or 'addToSeed' to specify)
      - Will use the following filters on reads:
         - Mapped length: restrict to range [0,500]. (parameter 'filterMappingLength')
         - Read length: keep all. (use 'filterReadLength' to limit)
         - Mapping quality: keep all. (use 'filterMQ' to limit)
         - Fragment length: keep all. (use 'filterFragmentLength' to

In [7]:
sums = {}
for i in range(n_reps):
    sums[i] = h2py.parsing.compute_h2_stats(
        vcf_file=f"data/mergedGPs.{i}.vcf.gz",
        use_genotype_probs=True,
        pop_file=pop_file,
        bed_file=bed_file,
        rec_map_file=rec_map_file,
        r_bins=r_bins,
        report=True,
    )
boot_data = h2py.parsing.bootstrap_data(sums)
model = h2py.H2stats.from_demes(g, sampled_demes=["pop0", "pop1"], u=u, r_bins=r_bins)
h2py.plotting.plot_h2_curves_comp(
    model,
    boot_data["means"],
    boot_data["varcovs"],
    r_bins=boot_data["bins"],
)

print(boot_data["means"][-1])
print(model.data[-1])

[26-07-14 15:01:39] Preparing data ...
[26-07-14 15:01:46] Prepared GenotypeProbMatrix (999982 sites, 1 samples, 1 pops)
[26-07-14 15:01:46] Loaded map coordinates (1e-08 to 0.01 M)
[26-07-14 15:01:46] Computing statistics and denominators ...
[26-07-14 15:01:46] Computed statistics and denominators.
[26-07-14 15:01:46]     Done!


/home/nick/Projects/h2py/h2py/parsing.py:417: RuntimeWarning: Degrees of freedom <= 0 for slice
  varcovs = [np.cov(np.array(m).T) for m in reshaped_means]


ValueError: deme pop1 is not in demography